In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/kylin0338/themeslabelled/themes_raw.csv.xlsx
/kaggle/input/datasets/kylin0338/negwithclusters/negative_with_clusters.csv
/kaggle/input/datasets/kylin0338/finalaspect/aspect_sentiment_final.csv


In [2]:
!pip install -q sentence-transformers
import pandas as pd, numpy as np
from sentence_transformers import SentenceTransformer

def make_key(d):
    return d["review_id"].astype(str) + "|" + d["aspect"] + "|" + d["sentence"]

# new negative set (same filter as before)
new = pd.read_csv("/kaggle/input/datasets/kylin0338/finalaspect/aspect_sentiment_final.csv")
neg = new[(new["sentiment_final"] == "negative") & (new["p_neg"] > 0.7)].reset_index(drop=True)
neg["key"] = make_key(neg)

# old clusters + your hand-written labels
old = pd.read_csv("/kaggle/input/datasets/kylin0338/negwithclusters/negative_with_clusters.csv")
old["key"] = make_key(old)
themes = pd.read_excel("/kaggle/input/datasets/kylin0338/themeslabelled/themes_raw.csv.xlsx")
old = old.merge(themes[["aspect", "cluster", "theme_label"]], on=["aspect", "cluster"], how="left")
old = old[(old["cluster"] != -1) & old["theme_label"].notna() & (old["theme_label"] != "")]

# carry labels over
neg = neg.merge(old[["key", "theme_label"]].drop_duplicates("key"), on="key", how="left")
print("carried over:", neg["theme_label"].notna().sum(), "| new, unlabeled:", neg["theme_label"].isna().sum(),
      "| old rows no longer present:", (~old["key"].isin(neg["key"])).sum())

carried over: 13915 | new, unlabeled: 211 | old rows no longer present: 1073


In [3]:
model = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")
emb = model.encode(neg["sentence"].tolist(), batch_size=256, normalize_embeddings=True, show_progress_bar=True)

MIN_SIM = 0.45          # below this, the sentence doesn't fit any theme well
neg["assigned"] = "carried"

for asp, g in neg.groupby("aspect"):
    known = g[g["theme_label"].notna() & (g["theme_label"] != "noise")]
    unk = g[g["theme_label"].isna()]
    if len(known) == 0 or len(unk) == 0:
        continue
    labels = known["theme_label"].unique()
    cents = np.stack([emb[known[known["theme_label"] == l].index].mean(axis=0) for l in labels])
    cents /= np.linalg.norm(cents, axis=1, keepdims=True)
    sims = emb[unk.index] @ cents.T
    best = sims.argmax(axis=1)
    neg.loc[unk.index, "theme_label"] = np.where(sims.max(axis=1) >= MIN_SIM, labels[best], "other")
    neg.loc[unk.index, "assigned"] = "nearest_theme"

print(neg["assigned"].value_counts())
print(neg[neg["assigned"] == "nearest_theme"]["theme_label"].value_counts().head(10))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/56 [00:00<?, ?it/s]

assigned
carried          13915
nearest_theme      211
Name: count, dtype: int64
theme_label
other                                             29
Performance: General lag and slow response        15
Battery: Poor battery life                        13
Camera: Camera malfunction or stopped working     10
Network: Wi-Fi connectivity and dropped calls      9
Screen: Damaged screen on delivery                 9
Camera: Poor camera quality                        9
Screen: Touchscreen responsiveness issues          8
Software: General dissatisfaction with Android     8
Software: Apps crashing and freezing               7
Name: count, dtype: int64


In [4]:
clean = neg[neg["theme_label"].notna() & ~neg["theme_label"].isin(["noise", "other", ""])].copy()

issues = (clean.groupby(["asin", "aspect", "theme_label"])
               .agg(mentions=("sentence", "count")).reset_index())
issues.to_csv("/kaggle/working/product_issues.csv", index=False)

examples = (clean.sort_values("p_neg", ascending=False)
                 .groupby(["aspect", "theme_label"]).head(5)
                 [["asin", "aspect", "theme_label", "sentence", "p_neg"]])
examples.to_csv("/kaggle/working/issue_examples.csv", index=False)

print(clean.groupby(["aspect", "theme_label"]).size().sort_values(ascending=False).head(25))

aspect            theme_label                                                    
network           Network: Carrier compatibility issues                              1072
battery           Battery: Poor battery life                                         1033
camera            Camera: Poor camera quality                                         579
battery           Battery: Defective or poor-performing battery                       569
customer_service  Customer Service: Return and replacement restrictions               462
network           Network: SIM card missing or not working                            456
screen            Screen: Malfunction or stopped working                              454
battery           Battery: Charging failure                                           438
screen            Screen: Damaged screen on delivery                                  428
                  Screen: Poor display quality                                        401
battery           

In [5]:
before = old.groupby(["aspect", "theme_label"]).size().rename("before")
after = clean.groupby(["aspect", "theme_label"]).size().rename("after")
cmp = pd.concat([before, after], axis=1).fillna(0).astype(int)
cmp["change_%"] = ((cmp["after"] - cmp["before"]) / cmp["before"].replace(0, np.nan) * 100).round(0)
print(cmp.sort_values("change_%").head(15))

                                                              before  after  \
aspect    theme_label                                                         
software  Software: Limited app availability and compatib...     414     57   
          Software: Apps crashing and freezing                   349     59   
          Software: General dissatisfaction with Android         352    142   
          Software: Problems caused by software updates          377    304   
usability Usability: Face unlock reliability and security...      22     20   
software  Software: Software bugs and update-related fixes       305    288   
screen    Screen: Scratches and physical damage                  391    368   
design    Design: Poor build quality and fragile design          116    113   
screen    Screen: Touchscreen responsiveness issues              294    285   
battery   Battery: Battery stops holding charge over time        397    391   
storage   Storage: Memory limitations despite expans

In [6]:
s = neg[neg["assigned"] == "nearest_theme"].sample(15, random_state=2)
for _, r in s.iterrows():
    print(f"[{r['theme_label']}]  {r['sentence'][:130]}")

[other]  Finger print reader isn't on par compared to my a1 and my camera app gives me a crash notification after I plug it in 75% of the t
[Battery: Poor battery life]  Out of the box, it's just right, but over the next year or two as apps become more ever more intensive and demanding with the batt
[Software: Apps crashing and freezing]  but the battery saver glitched the UI to oblivion
[other]  But any picture taken through an app like Snapchat or portrait mode will have that loud shutter sound.
[Screen: Malfunction or stopped working]  Samsung tech support advised to reset app settings...didn't help...list all the memorized emails and no resolution....
[Software: General dissatisfaction with Android]  However, the UI is too much worse than the Nexus 4 I have.
[Software: General dissatisfaction with Android]  Overall, the phone is designed creatively, but its hip design doesn't make up for its poor UI and lack of features that were essen
[Screen: Damaged screen on delivery]  I don't 